# FLUJOS

Construcción del flujo de caja libre para la firma del año base

$$FCFF = EBIT(1-t) - CapEx + D\&A - \Delta CT$$

El NOPAT ya se sabe o se puede obtener por el año base en traxion_anual.csv y de bitacora.md, vease:

**AÑO BASE**

Ingresos base: 38082.2

Margen operativo supuesto: 6.45%

EBIT base: 38,082.2 * 6.45% = 2,456.3


De esta forma, 

$$NOPAT = EBIT \times (1 - t_{\text{marginal}}) = 2{,}456.3 \times (1 - 30\%) = 1{,}719.4$$

La tasa del 30% es la marginal de Mexico proveniente de countrytaxrates.xls

Se necesita la reinversión para poder construir todo el FCFF, proposito de este notebook.


In [2]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
from src.datos import cargar_serie

# ano base
ingresos_base = 38082.2
ebit_base = 2456.3
tasa_mx = 0.30
crecimiento_organico = 0.032

wacc = 0.1234

# UDM 
capex_flota = 1881.1
capex_adquisiciones = 1541.2
dep_amort = 3111.4

nopat = ebit_base * (1 - tasa_mx)
print(f"NOPAT de año base: {nopat:,.1f}")

NOPAT de año base: 1,719.4


## Capex y depreciación

Primera decisión: si el capex normalizado incluye adquisiciones. La
metodología las cuenta como reinversión, porque comprar una empresa es una
forma de crecer igual que comprar camiones. Traxión gastó 1,541.2 en
adquirir Solistica dentro de la ventana de los últimos doce meses.

In [3]:
serie = cargar_serie("../data/interim/traxion_anual.csv")

serie["capex_total"] = serie["capex_flota"] + serie["capex_adquisiciones"].fillna(0)
serie["capex_sobre_ingresos"] = serie["capex_flota"] / serie["ingresos_totales"]
serie["da_sobre_ingresos"] = serie["dep_amort"] / serie["ingresos_totales"]
serie["capex_sobre_da"] = serie["capex_flota"] / serie["dep_amort"]

serie[["anio", "capex_flota", "capex_adquisiciones", "dep_amort",
       "capex_sobre_ingresos", "da_sobre_ingresos", "capex_sobre_da"]].round(3)

,anio,capex_flota,capex_adquisiciones,dep_amort,capex_sobre_ingresos,da_sobre_ingresos,capex_sobre_da
0,2021.0,1934.7,0.0,1503.1,0.113,0.088,1.287
1,2022.0,3390.1,1633.5,1914.5,0.167,0.094,1.771
2,2023.0,3434.0,61.3,2238.9,0.138,0.090,1.534
3,2024.0,3411.9,36.6,2512.0,0.117,0.086,1.358
4,2025.0,2348.5,1541.2,2869.4,0.069,0.085,0.818
5,2026.5,1881.1,1541.2,3111.4,0.049,0.082,0.605


El capex de flota cayó de 16.7% de los ingresos en 2022 a 4.9% en los últimos doce meses. La DyA se mantuvo estable en 8y9%. En la ventana más reciente Traxión invierte 60 centavos por cada peso que se deprecia.

Puede ser subinversión. Pero hay que ver qué compone esa DyA, porque no todo lo que se deprecia se repone con capex.


In [4]:
#nota 12 equipo de transporte, nota 13 intangibles y nota de arrendamientos del reporte anual 2025

da_2025 = pd.Series({
    "depreciacion_ppe": 1783.8,
    "depreciacion_derecho_uso": 920.7,
    "amortizacion_intangibles": 165.0,
})

print(da_2025)
print(f"\nTotal: {da_2025.sum():,.1f}")
print(f"D&A del estado de flujos 2025: 2,869.4")
print()
print(f"Capex de flota 2025: 2,348.5")
print(f"  contra D&A total          {2348.5/2869.4:.0%}")
print(f"  contra depreciacion de PPE {2348.5/1783.8:.0%}")

depreciacion_ppe            1783.8
depreciacion_derecho_uso     920.7
amortizacion_intangibles     165.0
dtype: float64

Total: 2,869.5
D&A del estado de flujos 2025: 2,869.4

Capex de flota 2025: 2,348.5
  contra D&A total          82%
  contra depreciacion de PPE 132%


La suma coincide con la DyA del estado de flujos.

**Un tercio de la DyA es de activos arrendados.** Esos no se reponen comprando: se reponen conmarrendamientos nuevos, y por eso no aparecen en el capex.

La amortización de intangibles tampoco exige reposición. 

Comparado contra la partida correcta, el capex de flota de 2025 supera en 32% la depreciación de los activos propios. **No hay subinversión.** 

In [5]:
activo_fijo = pd.DataFrame({
    "fecha": ["dic-2023", "dic-2024", "dic-2025", "jun-2026"],
    "equipo_transporte_neto": [14321.8, 15700.9, 16596.0, 16446.2],
    "derecho_de_uso_neto": [1386.3, 1166.3, 2061.6, 1947.4],
})
activo_fijo["total"] = activo_fijo["equipo_transporte_neto"] + activo_fijo["derecho_de_uso_neto"]
activo_fijo["variacion_pct"] = (activo_fijo["total"].pct_change() * 100).round(1)

activo_fijo.round(1)

#equipo de transporte neto de traxion_2025_anual.pdf ya que la tabla comparativa trae los tres cierres anuales; derecho de uso de la nota de arrendamientos, 
# donde el saldo al 1 de enero de 2024 sirve como cierre de 2023. Junio 2026 del 2T26.

,fecha,equipo_transporte_neto,derecho_de_uso_neto,total,variacion_pct
0,dic-2023,14321.8,1386.3,15708.1,NaN
1,dic-2024,15700.9,1166.3,16867.2,7.4
2,dic-2025,16596.0,2061.6,18657.6,10.6
3,jun-2026,16446.2,1947.4,18393.6,-1.4


a base de activos fijos creció 7.4% en 2024 y 10.6% en 2025, y solo cayo 1.4% en el primer semestre de 2026. No hay evidencia de que la empresa esté consumiendo su capacidad.

El salto del activo por derecho de uso entre 2024 y 2025, de 1,166.3 a
2,061.6, se debe a Solistica: la nota de
arrendamientos registra 1,094.2 de adiciones por adquisición de negocios.
Traxión no compró esos almacenes, los heredó arrendados.

**CAPITAL DE TRABAJO**

$$CT_{\text{no monetario}} = \left(CxC + \text{Inventarios} + \text{Otros activos op.}\right) - \left(\text{Proveedores} + \text{Acreedores} + \text{Impuestos por pagar} + \text{Otros pasivos op}\right)$$



In [6]:
balance = pd.DataFrame({
    "fecha": ["dic-2025", "jun-2026"],
    "cuentas_por_cobrar": [6874.131, 7329.079],
    "otras_cuentas_por_cobrar": [443.209, 478.514],
    "inventarios": [295.217, 370.044],
    "pagos_anticipados": [593.949, 754.634],
    "activos_impuestos": [255.336, 259.191],
    "otros_activos_impuestos": [560.914, 545.477],
    "proveedores": [3059.505, 3058.856],
    "acreedores": [1023.743, 966.969],
    "otros_impuestos_por_pagar": [1250.312, 1296.882],
    "pasivos_acumulados": [1608.679, 2269.195],
    "impuesto_utilidad": [108.568, 112.373],
    "ptu": [123.891, 76.499],
    "anticipos_clientes": [66.340, 12.950],
})

activos = ["cuentas_por_cobrar", "otras_cuentas_por_cobrar", "inventarios",
           "pagos_anticipados", "activos_impuestos", "otros_activos_impuestos"]
pasivos = ["proveedores", "acreedores", "otros_impuestos_por_pagar",
           "pasivos_acumulados", "impuesto_utilidad", "ptu", "anticipos_clientes"]

balance["activos_op"] = balance[activos].sum(axis=1)
balance["pasivos_op"] = balance[pasivos].sum(axis=1)
balance["ctno"] = balance["activos_op"] - balance["pasivos_op"]
balance["ctno_sobre_ingresos"] = balance["ctno"] / [33814.1, 38082.2]

balance[["fecha", "activos_op", "pasivos_op", "ctno", "ctno_sobre_ingresos"]].round(3)

,fecha,activos_op,pasivos_op,ctno,ctno_sobre_ingresos
0,dic-2025,9022.756,7241.038,1781.718,0.053
1,jun-2026,9736.939,7793.724,1943.215,0.051


Se excluye la caja del activo, porque es lo que se está midiendo. Y se excluye toda la deuda del pasivo, incluidas las obligaciones por arrendamiento: ya se contaron como deuda anteriormente. Los impuestos dif tampoco entran.

El capital de trabajo se mantiene en 5.1-5.3% de los ingresos.

El estado de flujos del 2T26 reporta una variación de -370 en el semestre,
contra los +161.5 que salen de comparar los dos balances. La empresa agrupa
las partidas de otra forma y su cifra incluye efectos cambiarios. Se usa el
cálculo propio, que sigue la definición de capital de trabajo no monetario.

In [7]:
#FCFF
ratio_ctno = balance["ctno_sobre_ingresos"].iloc[-1]
delta_ctno = ingresos_base * crecimiento_organico * ratio_ctno

capex_normalizado = capex_flota
depreciacion_reponible = dep_amort * (da_2025["depreciacion_ppe"] / da_2025.sum())
reinversion = capex_normalizado - depreciacion_reponible + delta_ctno

fcff = nopat - reinversion

print(f"NOPAT                          {nopat:>10,.1f}")
print(f"Capex de flota                 {-capex_normalizado:>10,.1f}")
print(f"Depreciacion reponible         {depreciacion_reponible:>10,.1f}")
print(f"Cambio en capital de trabajo   {-delta_ctno:>10,.1f}")
print(f"{'':31}{'-'*10}")
print(f"Reinversion                    {reinversion:>10,.1f}")
print(f"FCFF del ano base              {fcff:>10,.1f}")
print()
print(f"Tasa de reinversion            {reinversion/nopat:>10.1%}")

NOPAT                             1,719.4
Capex de flota                   -1,881.1
Depreciacion reponible            1,934.2
Cambio en capital de trabajo        -62.2
                               ----------
Reinversion                           9.1
FCFF del ano base                 1,710.3

Tasa de reinversion                  0.5%


## Resultado

| Concepto | Monto |
|---|---|
| NOPAT | 1,719.4 |
| Capex de flota | -1,881.1 |
| Depreciación reponible | +1,934.2 |
| Cambio en capital de trabajo | -62.2 |
| **Reinversión** | **9.1** |
| **FCFF del año base** | **1,710.3** |

Tasa de reinversión: 0.5%.

**La reinversión es prácticamente nula: 9.1 sobre un NOPAT de 1,719.4.** El capex de flota casi iguala la depreciación reponible, y el capital de trabajo consume poco porque el negocio no tiene inventarios relevantes.

Eso tiene una implicación directa. El crecimiento fundamental se estima como reinversión multiplicada por ROC, de modo que con una tasa de reinversión de 0.5% el crecimiento implícito es prácticamente cero.

Traxión no puede crecer sin invertir más de lo que invirtió en los últimos doce meses. Es coherente con lo anunciado: recorte de 500 millones de capex, salida del 25% de la flota de carga y migración a un modelo asset-light. La empresa está preservando caja, no expandiendo.